# 🔬 HTSPF — CIFAR-100 Vision Benchmark
**Google Colab GPU Runner**

This notebook runs the full CIFAR-100 benchmark grid:
- **5 HTSPF ablations** × 5 seeds = 25 runs  
- **3 vision baselines** (ResNet-18, ViT-Small, Perceiver-IO) × 5 seeds = 15 runs  
- **Total: 40 runs × 50 epochs**  
- **Est. wall time on T4:** ~4–6 hours

### Instructions
1. **Runtime → Change runtime type → GPU (T4)**
2. Run **Cell 1** (Setup) — clones repo, installs deps, mounts Drive
3. Run **Cell 2** (Config) — no changes needed
4. Run **Cell 3** (Ablations) — runs 25 experiments
5. Run **Cell 4** (Baselines) — runs 15 experiments
6. Run **Cell 5** (Verify & Download) — zips results to Drive

> ⚠️ **Colab disconnects after ~12 hours idle.** Results are saved to Google Drive after each run so nothing is lost on disconnect. Re-run only the cells that haven't finished.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Setup: Clone repo, install deps, mount Google Drive
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, subprocess, sys

# 1a. Mount Google Drive (results saved here, survive runtime resets)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/HTSPF_results/cifar100'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f'Drive output: {DRIVE_OUT}')

# 1b. Clone the repo
REPO_DIR = '/content/HTSPF'
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
    print('Repo updated.')
else:
    !git clone https://github.com/meetmehedi/HTSPF-Hierarchical-Temporal-Spatial-Pattern-Fusion.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print(f'Working dir: {os.getcwd()}')

# 1c. Install dependencies
!pip install -q ptflops sktime datasets scipy pyyaml

# 1d. Verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → GPU')

# 1e. Create local results dir (mirrored to Drive after each run)
LOCAL_OUT = os.path.join(REPO_DIR, 'results/raw')
os.makedirs(LOCAL_OUT, exist_ok=True)

# Copy any already-completed runs from Drive back (resume support)
import glob, shutil
existing = glob.glob(f'{DRIVE_OUT}/*.json')
for f in existing:
    dst = os.path.join(LOCAL_OUT, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
print(f'Resumed {len(existing)} previously completed runs from Drive.')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Config (no edits needed)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, glob, shutil

CONFIG   = 'configs/experiment.yaml'
LOCAL_OUT = os.path.join(REPO_DIR, 'results/raw')
DRIVE_OUT = '/content/drive/MyDrive/HTSPF_results/cifar100'
SEEDS     = [0, 1, 2, 3, 4]
ABLATIONS         = ['HTSPF_Full', 'HTSPF_noHCAA', 'HTSPF_noASG', 'HTSPF_noLDWT', 'HTSPF_noConflict']
VISION_BASELINES  = ['resnet18', 'vit_small', 'perceiver_io']
errors = []

def already_done(model, dataset, seed):
    """Skip experiments whose JSON already exists (resume support)."""
    return os.path.exists(os.path.join(LOCAL_OUT, f'{model}_{dataset}_seed{seed}.json'))

def run_experiment(model, dataset, seed):
    label = f'{model} | {dataset} | seed={seed}'
    if already_done(model, dataset, seed):
        print(f'  ⏭  Skipping {label} (already done)')
        return 0
    print(f'\n>>> {label}')
    ret = subprocess.run(
        [sys.executable, 'src/train.py',
         '--model', model, '--dataset', dataset,
         '--seed', str(seed), '--config', CONFIG,
         '--output', LOCAL_OUT],
        cwd=REPO_DIR
    ).returncode
    # Mirror result to Drive immediately after each run
    src_json = os.path.join(LOCAL_OUT, f'{model}_{dataset}_seed{seed}.json')
    if os.path.exists(src_json):
        shutil.copy(src_json, DRIVE_OUT)
        print(f'  💾 Saved to Drive')
    else:
        print(f'  ❌ ERROR: JSON not written for {label}')
        errors.append(label)
    return ret

total = (len(ABLATIONS) + len(VISION_BASELINES)) * len(SEEDS)
done  = len(glob.glob(f'{LOCAL_OUT}/*cifar100*.json'))
print(f'Progress: {done}/{total} CIFAR-100 experiments already done.')
print(f'Remaining: {total - done}')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — HTSPF Ablations (25 runs)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('='*60)
print('HTSPF ABLATIONS ON CIFAR-100')
print('='*60)

for model in ABLATIONS:
    for seed in SEEDS:
        run_experiment(model, 'cifar100', seed)

done_abl = len(glob.glob(f'{LOCAL_OUT}/HTSPF_*cifar100*.json'))
print(f'\n✅ Ablations complete — {done_abl}/25 files written')
print(f'   Errors so far: {errors if errors else "none"}')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Vision Baselines (15 runs)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('='*60)
print('VISION BASELINES ON CIFAR-100')
print('='*60)

for model in VISION_BASELINES:
    for seed in SEEDS:
        run_experiment(model, 'cifar100', seed)

done_base = len(glob.glob(f'{LOCAL_OUT}/*cifar100*.json'))
print(f'\n✅ Baselines complete — {done_base}/40 total CIFAR-100 files')
print(f'   Errors: {errors if errors else "none"}')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Verify results & create download zip
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, glob

all_json = sorted(glob.glob(f'{LOCAL_OUT}/*cifar100*.json'))
print(f'Total CIFAR-100 result files: {len(all_json)} (expected 40)\n')

for f in all_json:
    with open(f) as fh:
        d = json.load(fh)
    print(f'  {os.path.basename(f):55s}  acc={d.get("test_acc", 0):.2f}%')

# Zip everything on Drive for easy download
zip_path = '/content/drive/MyDrive/HTSPF_results/cifar100_results_FINAL'
shutil.make_archive(zip_path, 'zip', DRIVE_OUT)
print(f'\n📦 Zip saved to Drive: {zip_path}.zip')
print('Download from Google Drive → share or download to local machine.')
print('Then unzip into: results/raw/ in your local HTSPF repo.')
print('Then run: python scripts/compile_results.py')

if errors:
    print(f'\n⚠️  Failed experiments ({len(errors)}): re-run Cell 3/4 to retry.')
    for e in errors:
        print(f'  - {e}')
else:
    print('\n🎉 All 40 experiments completed with no errors!')
